In [1]:
# -*- coding: utf-8 -*-
"""
华泰期货《成交-持仓-价格三维视角下的期价走势影响研究》(流量立方体) 复现
运行环境: 聚宽研究环境 (Python 3.6 / pandas 0.23.4 / numpy 1.19.5)
说明:
  * 所有数据直接用 get_price 提取、直接在内存中使用, 不落盘
  * 所有结果直接 print, 不画图、不写文件
  * 价格取 主力连续(XXX9999); 成交量/持仓量/成交额取 品种指数(XXX8888, 全合约汇总),
    避免主力连续换月时持仓量跳变污染信号 (若 8888 无持仓量则回退到 9999)
"""
import datetime
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
try:
    from jqdata import *   # 研究环境里 get_price 已内置, 这里仅作保险
except Exception:
    pass

# ============================ 参数 ============================
START = '2024-01-01'
END = '2026-12-20'          # 超过今天自动截断到今天
WARMUP_DAYS = 200           # 向前多取的天数, 用于差分/流动性排名预热
TOP_N = 30                  # 流动性最好的品种数 (研究报告: 30 个)
LIQ_WEEKS = 12              # 流动性 = 过去 12 周(约 60 个交易日) 平均成交额, 逐周滚动排名(无未来函数)
COST = 0.0                  # 回测单边换手成本(占名义资金比例), 研究报告未考虑, 默认 0
ANN = 52                    # 周度年化因子
DAILY_START_AU = '2025-04-23'   # 黄金日度分析起点(研究报告 图10/11)

STATE_NAMES = ['SLB', 'WLB', 'SLL', 'WLL', 'SSB', 'WSB', 'SSC', 'WSC']
STATE_CN = {'SLB': '强多头建仓', 'WLB': '弱多头建仓', 'SLL': '强多头平仓', 'WLL': '弱多头平仓',
            'SSB': '强空头建仓', 'WSB': '弱空头建仓', 'SSC': '强空头平仓', 'WSC': '弱空头平仓'}

# 商品期货品种 (代码, 名称, 交易所); 金融期货(CCFX)与集运指数不纳入
VARIETIES = [
    ('A', '豆一', 'XDCE'), ('B', '豆二', 'XDCE'), ('M', '豆粕', 'XDCE'), ('C', '玉米', 'XDCE'),
    ('Y', '豆油', 'XDCE'), ('L', '聚乙烯', 'XDCE'), ('P', '棕榈油', 'XDCE'), ('V', 'PVC', 'XDCE'),
    ('J', '焦炭', 'XDCE'), ('JM', '焦煤', 'XDCE'), ('I', '铁矿石', 'XDCE'), ('JD', '鸡蛋', 'XDCE'),
    ('BB', '胶合板', 'XDCE'), ('FB', '纤维板', 'XDCE'), ('PP', '聚丙烯', 'XDCE'),
    ('CS', '玉米淀粉', 'XDCE'), ('EG', '乙二醇', 'XDCE'), ('RR', '粳米', 'XDCE'),
    ('EB', '苯乙烯', 'XDCE'), ('PG', '液化石油气', 'XDCE'), ('LH', '生猪', 'XDCE'),
    ('LG', '原木', 'XDCE'),
    ('SC', '原油', 'XINE'), ('NR', '20号胶', 'XINE'), ('LU', '低硫燃料油', 'XINE'),
    ('BC', '阴极铜', 'XINE'),
    ('AL', '铝', 'XSGE'), ('RU', '天然橡胶', 'XSGE'), ('FU', '燃料油', 'XSGE'),
    ('CU', '铜', 'XSGE'), ('ZN', '锌', 'XSGE'), ('AU', '黄金', 'XSGE'), ('WR', '线材', 'XSGE'),
    ('RB', '螺纹钢', 'XSGE'), ('PB', '铅', 'XSGE'), ('AG', '白银', 'XSGE'),
    ('BU', '沥青', 'XSGE'), ('HC', '热卷', 'XSGE'), ('SN', '锡', 'XSGE'), ('NI', '镍', 'XSGE'),
    ('SP', '纸浆', 'XSGE'), ('SS', '不锈钢', 'XSGE'), ('AO', '氧化铝', 'XSGE'),
    ('BR', '丁二烯橡胶', 'XSGE'),
    ('CF', '棉花', 'XZCE'), ('SR', '白糖', 'XZCE'), ('TA', 'PTA', 'XZCE'), ('OI', '菜籽油', 'XZCE'),
    ('FG', '玻璃', 'XZCE'), ('RS', '油菜籽', 'XZCE'), ('RM', '菜粕', 'XZCE'),
    ('MA', '甲醇', 'XZCE'), ('SF', '硅铁', 'XZCE'), ('SM', '锰硅', 'XZCE'),
    ('ZC', '动力煤', 'XZCE'), ('CY', '棉纱', 'XZCE'), ('AP', '苹果', 'XZCE'),
    ('CJ', '红枣', 'XZCE'), ('UR', '尿素', 'XZCE'), ('SA', '纯碱', 'XZCE'),
    ('PF', '短纤', 'XZCE'), ('PK', '花生', 'XZCE'), ('SH', '烧碱', 'XZCE'),
    ('PX', '对二甲苯', 'XZCE'), ('PR', '瓶片', 'XZCE'),
    ('RI', '早籼稻', 'XZCE'), ('LR', '晚籼稻', 'XZCE'), ('JR', '粳稻谷', 'XZCE'),
    ('WH', '强麦', 'XZCE'), ('PM', '普麦', 'XZCE'),
    ('SI', '工业硅', 'GFEX'), ('LC', '碳酸锂', 'GFEX'), ('PS', '多晶硅', 'GFEX'),
]
NAME = dict((v[0], v[1]) for v in VARIETIES)

TODAY_TS = pd.Timestamp(datetime.date.today())
START_TS = pd.Timestamp(START)
END_TS = min(pd.Timestamp(END), TODAY_TS)
END_S = END_TS.strftime('%Y-%m-%d')
FETCH_START_S = (START_TS - datetime.timedelta(days=WARMUP_DAYS)).strftime('%Y-%m-%d')


# ============================ 工具函数 ============================
def show_bars(title, labels, values, fmt='{:10.3f}', width=28):
    """用文本条形图代替画图"""
    print(title)
    vals = np.array(values, dtype=float)
    m = np.nanmax(np.abs(vals)) if np.isfinite(vals).any() else 1.0
    m = m if m > 0 else 1.0
    for l, v in zip(labels, vals):
        if np.isnan(v):
            print('  %-4s %s' % (l, '       nan'))
            continue
        n = int(round(abs(v) / m * width))
        print('  %-4s %s  %s' % (l, fmt.format(v), ('+' if v >= 0 else '-') * n))


def fetch(code):
    """提取日线; 失败返回 None"""
    for fields in (['close', 'volume', 'money', 'open_interest'], ['close', 'volume', 'money']):
        try:
            df = get_price(code, start_date=FETCH_START_S, end_date=END_S, frequency='daily',
                           fields=fields, skip_paused=True, panel=False)
        except Exception:
            continue
        if df is not None and len(df) > 0:
            if 'open_interest' not in df.columns:
                df['open_interest'] = np.nan
            return df
    return None


def load_variety(sym, exch):
    main = fetch('%s9999.%s' % (sym, exch))
    if main is None:
        return None
    idx = fetch('%s8888.%s' % (sym, exch))
    base = main
    if idx is not None and idx['open_interest'].fillna(0).sum() > 0:
        base = idx
    d = pd.DataFrame({'close': main['close']})
    d['volume'] = base['volume']
    d['oi'] = base['open_interest']
    d['money'] = base['money']
    d['money'] = d['money'].fillna(d['close'] * d['volume'])
    d = d.dropna(subset=['close', 'volume', 'oi'])
    d = d[d['volume'] > 0]
    if len(d) < 60:
        return None
    return d


def classify(p, o, v):
    """按研究报告 图3 的规则划分八个立方体状态 (p/o/v 为 价格/持仓/成交量 是否上升, numpy bool)"""
    return np.where(p & o & v, 'SLB',
           np.where(p & o & ~v, 'WLB',
           np.where(~p & ~o & v, 'SLL',
           np.where(~p & ~o & ~v, 'WLL',
           np.where(~p & o & v, 'SSB',
           np.where(~p & o & ~v, 'WSB',
           np.where(p & ~o & v, 'SSC', 'WSC')))))))


def build_states(d, freq='W'):
    """freq='W' 周度(每周最后一个交易日收盘价, 成交量周内求和, 持仓量取周末); freq='D' 日度"""
    if freq == 'W':
        x = d.resample('W-FRI').agg({'close': 'last', 'volume': 'sum', 'oi': 'last', 'money': 'sum'})
        x = x.dropna(subset=['close'])
        x = x[x['volume'] > 0]
        if len(x) > 0 and x.index[-1] > TODAY_TS:     # 剔除尚未走完的当周
            x = x.iloc[:-1]
    else:
        x = d[['close', 'volume', 'oi', 'money']].copy()
    o = pd.DataFrame(index=x.index)
    o['close'] = x['close']
    o['money'] = x['money']
    o['dp'] = x['close'].diff()
    o['ret'] = x['close'].pct_change()
    doi = x['oi'].diff()
    dv = x['volume'].diff()
    valid = o['dp'].notnull() & doi.notnull() & dv.notnull()
    st = classify((o['dp'] > 0).values, (doi > 0).values, (dv > 0).values)
    o['state'] = pd.Series(st, index=o.index).where(valid)
    o['next_dp'] = o['dp'].shift(-1)
    o['next_ret'] = o['ret'].shift(-1)
    return o


# ============================ 数据提取 ============================
print('=' * 70)
print('数据提取: %s ~ %s (含预热期自 %s 起)' % (START, END_S, FETCH_START_S))
daily = {}
failed = []
for sym, nm, ex in VARIETIES:
    d = load_variety(sym, ex)
    if d is None:
        failed.append(sym)
    else:
        daily[sym] = d
print('成功品种 %d 个; 无数据/数据过少而跳过: %s' % (len(daily), ','.join(failed)))
syms = sorted(daily.keys())

W = {}
for s in syms:
    W[s] = build_states(daily[s], 'W')

# 逐周流动性排名 (过去 LIQ_WEEKS 周平均成交额, 取前 TOP_N)
money_w = pd.DataFrame(dict((s, W[s]['money']) for s in syms), columns=syms)
liq = money_w.rolling(LIQ_WEEKS, min_periods=4).mean()
top = liq.rank(axis=1, ascending=False) <= TOP_N

frames = []
for s in syms:
    f = W[s].copy()
    f['sym'] = s
    f['top'] = top[s].reindex(f.index).fillna(False).astype(bool)
    frames.append(f)
long_all = pd.concat(frames)
long_win = long_all[long_all.index >= START_TS]       # 样本区间内的全部周度观测
print('周度观测数(样本区间内): %d, 覆盖周数: %d' % (len(long_win), long_win.index.nunique()))
print('注意: 样本区间只有约 %.1f 年, 统计结果的样本量远小于研报的 2000-2025 年, 结论仅供参考。'
      % ((END_TS - START_TS).days / 365.0))

# ============================ 一、解释性: 全品种各状态 夏普/收益/波动 (图4-6) ============================
print('\n' + '=' * 70)
print('一、全品种各立方体状态: 年化收益率 / 年化波动 / 夏普比率 (图4、图5、图6)')
print('   (口径: 状态与当期收益同期, 即"提前预知涨跌", 收益按方向调整后取绝对值, 周度年化)')
L = long_win.dropna(subset=['state', 'ret']).copy()
L['absret'] = L['ret'].abs()
g = L.groupby('state')['absret']
res = pd.DataFrame({'年化收益率': g.mean() * ANN, '年化波动': g.std() * np.sqrt(ANN), '样本数': g.count()})
res['夏普比率'] = res['年化收益率'] / res['年化波动']
res = res.reindex(STATE_NAMES)
print(res.round(3).to_string())
r4 = res['夏普比率'].sort_values(ascending=False)
show_bars('\n图4 各状态夏普比率 (降序):', list(r4.index), r4.values, '{:10.2f}')
r5 = res['年化收益率'].sort_values(ascending=False)
show_bars('\n图5 各状态平均年化收益率:', list(r5.index), r5.values, '{:10.3f}')
r6 = res['年化波动'].sort_values(ascending=False)
show_bars('\n图6 各状态平均年化波动:', list(r6.index), r6.values, '{:10.3f}')

oi_up = ['SLB', 'WLB', 'SSB', 'WSB']
vol_up = ['SLB', 'SLL', 'SSB', 'SSC']
print('\n核心结论检验:')
print('  持仓量增加的四个状态 平均夏普 = %.2f, 其余四个状态 = %.2f'
      % (res.loc[oi_up, '夏普比率'].mean(), res.loc[[s for s in STATE_NAMES if s not in oi_up], '夏普比率'].mean()))
print('  成交量增加的四个状态 平均年化收益率 = %.3f, 其余四个状态 = %.3f'
      % (res.loc[vol_up, '年化收益率'].mean(), res.loc[[s for s in STATE_NAMES if s not in vol_up], '年化收益率'].mean()))

# ============================ 二、解释性: 黄金/螺纹钢/铁矿石 (图7-17) ============================
def analyze_product(sym, label):
    if sym not in W:
        print('\n[%s] 无数据, 跳过' % label)
        return
    w = W[sym][W[sym].index >= START_TS]
    px = w['close']
    print('\n--- %s期货主连 (%s) 周度 ---' % (label, sym))
    print('区间 %s ~ %s | 首周收盘 %.2f | 末周收盘 %.2f | 最高 %.2f | 最低 %.2f | 区间涨跌 %.2f 元(%.1f%%)'
          % (px.index[0].strftime('%Y-%m-%d'), px.index[-1].strftime('%Y-%m-%d'), px.iloc[0], px.iloc[-1],
             px.max(), px.min(), px.iloc[-1] - px.iloc[0], (px.iloc[-1] / px.iloc[0] - 1) * 100))
    cnt = w['state'].value_counts().reindex(STATE_NAMES).fillna(0).astype(int)
    contrib = w.groupby('state')['dp'].sum().reindex(STATE_NAMES).fillna(0.0)
    tb = pd.DataFrame({'状态数量(次)': cnt, '当期价格变化总量(元)': contrib})
    print(tb.round(2).to_string())
    show_bars('各状态数量:', STATE_NAMES, cnt.values, '{:6.0f}', 24)
    show_bars('各状态当期价格变化总量(元):', STATE_NAMES, contrib.values, '{:10.2f}', 24)
    print('状态数量最多: %s(%s); 价格上涨贡献最大: %s; 价格下行压力最大: %s'
          % (cnt.idxmax(), STATE_CN[cnt.idxmax()], contrib.idxmax(), contrib.idxmin()))


print('\n' + '=' * 70)
print('二、立方体对价格走势的解释 (图7-17)')
analyze_product('AU', '黄金')
analyze_product('RB', '螺纹钢')
analyze_product('I', '铁矿石')

print('\n--- 黄金期货主连: %s 之后 日度立方体 (图10、图11) ---' % DAILY_START_AU)
if 'AU' in daily:
    dd = build_states(daily['AU'], 'D')
    dd = dd[dd.index >= pd.Timestamp(DAILY_START_AU)]
    cnt = dd['state'].value_counts().reindex(STATE_NAMES).fillna(0).astype(int)
    contrib = dd.groupby('state')['dp'].sum().reindex(STATE_NAMES).fillna(0.0)
    print('交易日数 %d | 区间涨跌 %.2f 元' % (len(dd), dd['close'].iloc[-1] - dd['close'].iloc[0]))
    print(pd.DataFrame({'状态数量(日度)': cnt, '当期价格变化总量(元)': contrib}).round(2).to_string())
    show_bars('各状态数量(日度):', STATE_NAMES, cnt.values, '{:6.0f}', 24)
    show_bars('各状态当期价格变化总量(元):', STATE_NAMES, contrib.values, '{:10.2f}', 24)

# ============================ 三、预测性 (图18、图19) ============================
print('\n' + '=' * 70)
print('三、立方体对未来一期价格/收益的预测 (图18、图19)')
LT = long_win[long_win['top']].dropna(subset=['state', 'next_ret']).copy()
print('样本: 每周流动性前 %d 品种, 共 %d 个观测' % (TOP_N, len(LT)))
g = LT.groupby('state')['next_ret']
r18 = pd.DataFrame({'下期平均收益率(%)': g.mean() * 100, '标准误(%)': g.std() / np.sqrt(g.count()) * 100,
                    '样本数': g.count()}).reindex(STATE_NAMES)
r18['t值'] = r18['下期平均收益率(%)'] / r18['标准误(%)']
print(r18.round(3).to_string())
show_bars('\n图18 全品种各状态对下期收益率的影响(%):', STATE_NAMES, r18['下期平均收益率(%)'].values, '{:10.3f}', 28)

print('\n图19 部分品种处于不同状态下未来一期平均价格变化(元):')
rows = {}
for sym in ['SC', 'M', 'TA', 'AU', 'I', 'A']:
    if sym in W:
        w = W[sym][W[sym].index >= START_TS].dropna(subset=['state', 'next_dp'])
        rows['%s(%s)' % (NAME[sym], sym)] = w.groupby('state')['next_dp'].mean().reindex(STATE_NAMES)
if rows:
    print(pd.DataFrame(rows).T.reindex(columns=STATE_NAMES).round(2).to_string())

# ============================ 四、策略回测 (图20、图21) ============================
print('\n' + '=' * 70)
print('四、流量立方体策略回测 (图20、图21)')
print('规则: 上周状态为 SLB/SSC -> 本周做多; SSB/SLL -> 本周做空; 其余空仓;')
print('      仅交易当周流动性前 %d 品种, 每品种分配 1/%d 资金; 周度换仓; 单边成本 COST=%.4f' % (TOP_N, TOP_N, COST))

state_w = pd.DataFrame(dict((s, W[s]['state']) for s in syms), columns=syms)
nextret_w = pd.DataFrame(dict((s, W[s]['next_ret']) for s in syms), columns=syms)
sig_map = {'SLB': 1, 'SSC': 1, 'SSB': -1, 'SLL': -1}
pos = state_w.applymap(lambda s: sig_map.get(s, 0))
pos = pos.where(top, 0)
gross = (pos * nextret_w).sum(axis=1) / float(TOP_N)
turnover = pos.diff().abs().sum(axis=1) / float(TOP_N)
pnl = gross - turnover * COST
strat = pnl.shift(1)                      # t 周信号 -> 收益记在 t+1 周
strat = strat[(strat.index >= START_TS) & (strat.index <= END_TS)].dropna()


def perf(r):
    n = len(r)
    if n < 2:
        return None
    ann = r.mean() * ANN
    vol = r.std() * np.sqrt(ANN)
    eq = (1 + r).cumprod()
    return {'周数': n, '年化收益率(%)': ann * 100,
            '复利年化(%)': ((1 + r).prod() ** (ANN / float(n)) - 1) * 100,
            '年化波动(%)': vol * 100, '年化夏普': ann / vol if vol > 0 else np.nan,
            '最大回撤(%)': (eq / eq.cummax() - 1).min() * 100}


if len(strat) < 2:
    print('回测样本不足')
else:
    out = {}
    for y, gr in strat.groupby(strat.index.year):
        p = perf(gr)
        if p:
            out[str(y)] = p
    out['回测期间'] = perf(strat)
    tb = pd.DataFrame(out).T[['周数', '年化收益率(%)', '复利年化(%)', '年化波动(%)', '年化夏普', '最大回撤(%)']]
    print('\n图21 分年度/全期表现 (年化夏普比率):')
    print(tb.round(2).to_string())
    full = out['回测期间']
    print('\n回测期间: 年化收益率 %.2f%% (复利 %.2f%%), 年化夏普 %.2f, 最大回撤 %.2f%%'
          % (full['年化收益率(%)'], full['复利年化(%)'], full['年化夏普'], full['最大回撤(%)']))
    show_bars('\n图21 各年度年化夏普:', list(tb.index), tb['年化夏普'].values, '{:8.2f}', 24)

    cum = strat.cumsum() * 100
    cq = cum.resample('Q').last()
    show_bars('\n图20 累计收益率(%, 单利累加, 按季度末):',
              [d.strftime('%Y-%m') for d in cq.index], cq.values, '{:9.2f}', 28)
    print('\n最新一周各品种持仓信号 (+1多/-1空, 仅列出非空仓):')
    last = pos.iloc[-1]
    last = last[last != 0]
    print(', '.join('%s%s(%+d)' % (NAME[s], s, int(v)) for s, v in zip(last.index, last.values)) or '无')
    tl = top.iloc[-1]
    print('\n最新一周流动性前%d品种: %s' % (TOP_N, ','.join(NAME[s] for s in tl[tl].index)))

print('\n' + '=' * 70)
print('运行结束 (全部结果已在上方打印, 未写入任何文件)')


数据提取: 2024-01-01 ~ 2026-10-29 (含预热期自 2023-06-15 起)
成功品种 67 个; 无数据/数据过少而跳过: ZC,RI,LR,JR,WH,PM
周度观测数(样本区间内): 9566, 覆盖周数: 145
注意: 样本区间只有约 2.8 年, 统计结果的样本量远小于研报的 2000-2025 年, 结论仅供参考。

一、全品种各立方体状态: 年化收益率 / 年化波动 / 夏普比率 (图4、图5、图6)
   (口径: 状态与当期收益同期, 即"提前预知涨跌", 收益按方向调整后取绝对值, 周度年化)
       年化收益率   年化波动   样本数   夏普比率
state                           
SLB    1.591  0.253  1463  6.291
WLB    1.178  0.223   912  5.286
SLL    1.224  0.183   963  6.705
WLL    1.012  0.157  1640  6.461
SSB    1.224  0.164  1274  7.462
WSB    1.109  0.144  1042  7.680
SSC    1.375  0.225   978  6.109
WSC    0.843  0.145  1290  5.828

图4 各状态夏普比率 (降序):
  WSB        7.68  ++++++++++++++++++++++++++++
  SSB        7.46  +++++++++++++++++++++++++++
  SLL        6.70  ++++++++++++++++++++++++
  WLL        6.46  ++++++++++++++++++++++++
  SLB        6.29  +++++++++++++++++++++++
  SSC        6.11  ++++++++++++++++++++++
  WSC        5.83  +++++++++++++++++++++
  WLB        5.29  +++++++++++++++++++

图5 各状态平均年化收益率:
  SLB       1.5